# Matrix multiplication with Strassen's algorithm
In this notebook we will look at an efficient algorithm for multiplying large square matrices. This is required for example in certain graph-theory algorithms: as we saw in one of the tickables, the structure of a graph or network can be described as an adjacency matrix. It is not uncommon for these networks to have a very large number of nodes (think of social networks such as facebook). As we will see, the computational complexity of the naive method for multiplying two $n\times n$ matrices is $\mathcal{O}(n^3)$, in other words, the cost grows with the third power of the problem size $n$. It turns out that there is a more efficient method based on the divide-and-conquer approach, which reduces the computational complexity to $\mathcal{O}(n^{2.8074})$. For large values of $n$ we expect this method to be faster than the naive approach. We will describe the improved algorithm due to Strassen and discuss its computational complexity here. The topic is also investigated in detail in **chapter 4.2 of "Introduction to Algorithms" by Cormen, Leiserson, Rivest and Stein**.

<hr style="height: 2px">

### What you will learn
In this notebook we will cover the following topics:

* Number of elops in matrix-matrix multiplication
* A first attempt at a divide-and-conquer approach
* Strassen's algorithm
* Computational complexity and further improvements

<hr style="height: 2px">

*&#169; Eike Mueller, University of Bath 2019-2023. These lecture notes are copyright of Eike Mueller, University of Bath. They are provided exclusively for educational purposes at the University and are to be downloaded or copied for your private study only. Further distribution, e.g. by upload to external repositories, is prohibited.*

## Matrix-matrix multiplication
The problem that we are concerned with here is the multiplication of two (large) $n\times n$ matrices $A$ and $B$. 
The product of two matrices has been defined in the [notebook on vectors and matrices](../week01/Lecture_01a_VectorsMatrices.ipynb), but let us repeat the key points here. To distinguish the product of two matrices from the product of two numbers, we will denote it by an empty circle (''$\circ$'').

Given two $n\times n$ matrices $A$, $B$ we want to compute

$$
C = A \circ B.
$$

or, more explicitly:

$$
\begin{pmatrix}
C_{00} & C_{01} & C_{02} & \ldots & C_{0,n-1}\\
C_{10} & C_{11} & C_{12} & \ldots & C_{1,n-1}\\
C_{20} & C_{21} & C_{12} & \ldots & C_{2,n-1}\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
C_{n-1,0} & C_{n-1,1} & C_{n-1,2} & \ldots & C_{n-1,n-1}
\end{pmatrix}
=
\begin{pmatrix}
A_{00} & A_{01} & A_{02} & \ldots & A_{0,n-1}\\
A_{10} & A_{11} & A_{12} & \ldots & A_{1,n-1}\\
A_{20} & A_{21} & A_{22} & \ldots & A_{2,n-1}\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
A_{n-1,0} & A_{n-1,1} & A_{n-1,2} & \ldots & A_{n-1,n-1}
\end{pmatrix}
\circ
\begin{pmatrix}
B_{00} & B_{01} & B_{02} & \ldots & B_{0,n-1}\\
B_{10} & B_{11} & B_{12} & \ldots & B_{1,n-1}\\
B_{20} & B_{21} & B_{22} & \ldots & B_{2,n-1}\\
\vdots & \vdots & \vdots & \ddots & \vdots\\
B_{n-1,0} & B_{n2} & B_{n-1,2} & \ldots & B_{n-1,n-1}
\end{pmatrix}
$$

Recall that in line with the Python numbering convention, we start counting at zero.

The matrix $C$ also has $n\times n$ entries. The entry $C_{ij}$ in row $i$ and column $j$ is given by the dot product of the $i$-th row of $A$ with the $j$-th column of $B$; more specifically:

$$
C_{ij} = \sum_{k=0}^{n-1} A_{ik}\cdot B_{kj} \qquad\text{for all $i,j=0,\dots,n-1$}
$$

Each of the terms in the sum is the product of two numbers, denoted by a small, empty dot (''$\cdot$'').

### Number of elops
How many elops are required to compute all $n^2$ entries of the $n\times n$ matrix $C$? To work this out, let us write down the pseudocode for matrix-matrix multiplication:

### Algorithm: naive matrix-matrix multiplication
**Input**: $n\times n$ matrices $A$ and $B$ **Output**: product $C=A\circ B$
1. **for** $i=0,1,2,\dots,n-1$ **do**
1. $~~~~$ **for** $j=0,1,2,\dots,n-1$ **do**
1. $~~~~~~~~$ Set $C_{ij}\mapsto 0$
1. $~~~~~~~~$ **for** $k=0,1,2,\dots,n-1$ **do**
1. $~~~~~~~~~~~~$ Set $C_{ij} \mapsto C_{ij} + A_{ik}\cdot B_{kj}$
1. $~~~~~~~~$ **end for**
1. $~~~~$ **end for**
1. **end for**

The loop in lines 4-6 contains $n$ additions and $n$ multiplications. Since this loop is executed $n^2$ times (by looping over $i$ and $j$ in the two outer loops), the total number of elops is

$$
E_{\text{matrix-matrix}}^{\text{naive}}(n) = 2n^3 = \Theta(n^3)
$$

In other words, the number of elops grows with the third power of the problem size $n$. To put this into perspective, note that a modern CPU (the central processing unit in your computer) can carry out between $10^{10}$ and $10^{11}$ elops per second. Multiplying two matrices of size $10,000 \times 10,000$ requires $2\cdot 10^{12}$ elops and will take between 20 and 200 seconds. There are many applications where you need to multiply matrices that are larger than this: computing the product of two matrices with $n=10^6$ (=1 million) would take more than 100 million years!

## Divide-and-conquer
Can we do better than the naive algorithm? How about using a divide-and-conquer approach, which reduces the problem to smaller subproblems?
### First attempt
To simplify the following discussion, we make the assumption that $n$ is an integer power of two, i.e. $n=2^{k}$ for $k\in\mathbb{N}$. We can now write

$$
\begin{pmatrix}
C_{00} & C_{01} \\
C_{10} & C_{11}
\end{pmatrix}
=
\begin{pmatrix}
A_{00} & A_{01} \\
A_{10} & A_{11}
\end{pmatrix}
\circ
\begin{pmatrix}
B_{00} & B_{01} \\
B_{10} & B_{11}
\end{pmatrix}
$$

where, in a slight above of notation, each of the matrices $A_{ij}$, $B_{ij}$ and $C_{ij}$ is a $\frac{n}{2}\times \frac{n}{2}$ matrix. More explicitly, we have that

$$
\begin{aligned}
C_{00} &= A_{00}\circ B_{00} + A_{01} \circ B_{10},\\
C_{01} &= A_{00}\circ B_{01} + A_{01} \circ B_{11},\\
C_{10} &= A_{10}\circ B_{00} + A_{11} \circ B_{10},\\
C_{11} &= A_{10}\circ B_{01} + A_{11} \circ B_{11}.\\
\end{aligned}\qquad(\dagger)
$$

Each of the products on the right hand side is the product of two $\frac{n}{2}\times\frac{n}{2}$ matrices. The key idea is now to apply the method recursively: to compute the products $A_{ik}\circ B_{kj}$ of the $\frac{n}{2}\times \frac{n}{2}$ matrices in the fours equations in Eq. ($\dagger$), we split each of the matrices $A_{00}, B_{01}$ etc. into smaller $\frac{n}{4}\times \frac{n}{4}$. This process can be repeated recursively until eventually we will obtain matrices of size $1\times 1$, which can be multiplied like ordinary numbers.

The following recursive pseudocode illustrates this idea by defining a recursive function $\text{rmat_mul()}$:

### Algorithm: recursive matrix-matrix multiplication $C=\text{rmat_mul}(A,B)$
**Input**: $n\times n$ matrices $A$ and $B$ **Output**: product $C=A\circ B$
1. **if** $n==1$ **then**
1. $~~~~$ $C_{00}\mapsto A_{00}\cdot B_{00}$
1. **else**
1. $~~~~$ $C_{00} = \text{rmat_mul}(A_{00},B_{00}) + \text{rmat_mul}(A_{01},B_{10})$
1. $~~~~$ $C_{01} = \text{rmat_mul}(A_{00},B_{01}) + \text{rmat_mul}(A_{01},B_{11})$
1. $~~~~$ $C_{10} = \text{rmat_mul}(A_{10},B_{00}) + \text{rmat_mul}(A_{11},B_{10})$
1. $~~~~$ $C_{11} = \text{rmat_mul}(A_{10},B_{01}) + \text{rmat_mul}(A_{11},B_{11})$
1. **end if**

### Computational complexity
How many elops are required to execute this algorithm? If $n\ge 2$, we need to carry out the following operations:

* Add two $\frac{n}{2}\times\frac{n}{2}$ matrices. This has to be done four times, and requires $4\left(\frac{n}{2}\right)^2=n^2$ additions.
* Multiply two $\frac{n}{2}\times\frac{n}{2}$ matrices through recursive calls of the function to itself; this has to be done eight times.

If $n=1$ we only have to multiply two numbers, which costs one elop. Let $E_{\text{matrix-matrix}}^{\text{recursive}}(n)$ be the number of elops required to execute the above algorithm. Using the above two observations we find that

$$
E_{\text{matrix-matrix}}^{\text{recursive}}(n) = \begin{cases}
8E_{\text{matrix-matrix}}^{\text{recursive}}\left(\frac{n}{2}\right) + n^2 & \text{if $n\ge 2$}\\
1 & \text{if $n=1$}
\end{cases}\qquad(\ddagger)
$$

This expression defines $E_{\text{matrix-matrix}}^{\text{recursive}}(n)$ recursively. It can be shown that

$$
E_{\text{matrix-matrix}}^{\text{recursive}}(n) = (2n-1)n^2 = \Theta(n^3).
$$

(You can double check this by inserting the expression for $E_{\text{matrix-matrix}}^{\text{recursive}}(n)$ into Eq. ($\ddagger$)).

Although compared to the naive algorithm we save $n^2$ operations, this does not really matter for large values of $n$ where $E_{\text{matrix-matrix}}^{\text{recursive}}(n)\approx 2n^3=E_{\text{matrix-matrix}}^{\text{naive}}(n)$. Most importantly, the computational cost is still $\Theta(n^3)$, i.e. grows with the third power of the problem size $n$. Our naive divide-and-conquer approach has not helped much here.

<a id='strassen_algorithm'></a>
### Strassen's algorithm
The key problem with the divide-and-conquer algorithm described in the previous section is that we need eight recursive calls to the function itself. Strassen observed that the number of recursive calls can be reduced from eight to seven by defining the following seven matrix products:

$$
\begin{aligned}
P_0 &= A_{00}\circ(B_{01}-B_{11})\\
P_1 &= (A_{00}+A_{01})\circ B_{11}\\
P_2 &= (A_{10}+A_{11})\circ B_{00}\\
P_3 &= A_{11}\circ(B_{10}-B_{00})\\
P_4 &= (A_{00}+A_{11})\circ (B_{00}+B_{11})\\
P_5 &= (A_{01}-A_{11})\circ(B_{10}+B_{11})\\
P_6 &= (A_{00}-A_{10})\circ(B_{00}+B_{01})
\end{aligned}\qquad(\star)
$$

Some tedious, but straightforward algebra now shows that

$$
\begin{aligned}
C_{00} &= P_4+P_3-P_1+P_5\\
C_{01} &= P_0 + P_1\\
C_{10} &= P_2 + P_3\\
C_{11} &= P_4 + P_0 - P_2 - P_6
\end{aligned}\qquad{(\triangleleft)}
$$

The key trick is now to carry out the seven products of $\frac{n}{2}\times \frac{n}{2}$ matrices in Eq. ($\star$) recursively. We can write down Strassen's algorithm in pseudocode as follows:

### Algorithm: Strassen's method for matrix-matrix multiplication $C=\text{strassen}(A,B)$
**Input**: $n\times n$ matrices $A$ and $B$ **Output**: product $C=A\circ B$
1. **if** $n==1$ **then**
1. $~~~~$ $C_{00}\mapsto A_{00}\cdot B_{00}$
1. **else**
1. $~~~~$ $P_0 = \text{strassen}(A_{00},B_{01}-B_{11})$
1. $~~~~$ $P_1 = \text{strassen}(A_{00}+A_{01},B_{11})$
1. $~~~~$ $\dots$ (see Eq. ($\star$) above)
1. $~~~~$ $P_6 = \text{strassen}(A_{00}-A_{10},B_{00}+B_{01})$
1. $~~~~$ $C_{00} = P_4+P_3-P_1+P_5$
1. $~~~~$ $C_{01} = P_0 + P_1$
1. $~~~~$ $C_{10} = P_2 + P_3$
1. $~~~~$ $C_{11} = P_4 + P_0 - P_2 - P_6$
1. **end if**

Most importantly, the algorithm only contains *seven* recursive calls of the function $\text{strassen}()$ to itself to compute the $\frac{n}{2}\times \frac{n}{2}$ matrices $P_0,P_1,P_2,\dots,P_6$.

<a id='strassen_complexity'></a>
### Computational complexity
Again, we are interested in the number of elops that are required to execute the function $\text{strassen}()$ for a particular value of $n$. If $n\ge 2$ we need to carry out the following operations:

* Add 18 matrices of size $\frac{n}{2}\times\frac{n}{2}$, this requires $18\left(\frac{n}{2}\right)^2 = \frac{9}{2}n^2$ elops.
* Multiply seven matrices of size $\frac{n}{2}\times \frac{n}{2}$ with recursive calls of the function to itself with $\frac{n}{2}$.

For $n=1$ we again only need a single elop. This allows us to write down a recursive expression for the number of elops for a given problem size $n$:

$$
E_{\text{matrix-matrix}}^{\text{strassen}}(n) = \begin{cases}
7E_{\text{matrix-matrix}}^{\text{strassen}}\left(\frac{n}{2}\right) + \frac{9}{2}n^2 & \text{for $n\ge 2$}\\
1 & \text{for $n=1$}
\end{cases}\qquad(\diamond)
$$

It is possible to show that this recursion is solved by

$$
E_{\text{matrix-matrix}}^{\text{strassen}}(n) = 7\cdot n^{\log_2(7)} -6n^2 = \Theta(n^{\log_2(7)}) = \Theta(n^{2.8074})\qquad\text{with $2<\log_2(7)\le 2.8074<3$}
$$

(Again, you should convince yourself that this is true by inserting the expression for $E_{\text{matrix-matrix}}^{\text{strassen}}(n)$ into Eq. ($\diamond$); bear in mind that $2^{-\log_2(7)}=\frac{1}{7}$.)

Hence, for large values of $n$ the number of elops grows with a power of $n$ which is smaller than three. For large values of $n$ we expect Strassen algorithm to be faster than the naive matrix-matrix multiplication method described at the beginning of this notebook.

In our entire derivation we have assumed that $n$ is a power of two. If this is not the case, one can still apply the recursion by Strassen as long as $n$ is even, and then fall back to the naive matrix-matrix multiplication if $n$ is odd. Alternatively, one could pad the matrices $A$ and $B$ with zeros to obtain matrices of size $2^k\times 2^k$ for some integer $k$.

While for large values of $n$ Strassen's algorithm will eventually outperform the naive method for matrix-matrix multiplication, it is not so obvious for which matrix sizes this actually happens. To answer this question, we need to compare the actual number of elops, not just the exponent of $n$ in big-O notation. For this, we plotted both $E_{\text{matrix-matrix}}(n)$ (the number of elops for the naive matrix-matrix multiplication) and $E_{\text{matrix-matrix}}^{\text{strassen}}(n)$ in the following figure:

![elops for matrix-matrix multiplication](./figures/CostMatMatMult.png "Number of elops for naive matrix-matrix multiplication (red) and Strassen's method (blue)")

As the plot shows, the crossover point is at around $n=1000$, so we exect that for matrices that are larger than this it pays off to use Strassen's algorithm. In reality, the exact position of the crossover point also depends somewhat on the implementation and the computer hardware that we use. However, a further discussion of these subtleties is beyond the scope of this course.

## Implementation
The following Python code implements Strassen's algorithm for matrix-matrix multiplication:

In [1]:
import numpy as np

def strassen(A,B):
    '''Strassen s algorithm for multiplying two square n x n matrices A, B to
    obtain C = AB. It is assumed that n = 2^k, i.e. that n is a power of two.
    
    input: A, B: two n x n matrices
    output: matrix-product C = AB
    '''
    n = A.shape[0] # Work out matrix size
    if n==1:
        return A@B
    else:
        # Extract submatrices A_{00}, A_{01} etc. using slicing
        A_00 = A[:n//2,:n//2]
        A_01 = A[:n//2,n//2:]
        A_10 = A[n//2:,:n//2]
        A_11 = A[n//2:,n//2:]
        B_00 = B[:n//2,:n//2]
        B_01 = B[:n//2,n//2:]
        B_10 = B[n//2:,:n//2]
        B_11 = B[n//2:,n//2:]
        # Construct matrices P_0, P_1, ..., P_6 via recursive calls
        P_0 = strassen(A_00,B_01-B_11)
        P_1 = strassen(A_00+A_01,B_11)
        P_2 = strassen(A_10+A_11,B_00)
        P_3 = strassen(A_11,B_10-B_00)
        P_4 = strassen(A_00+A_11,B_00+B_11)
        P_5 = strassen(A_01-A_11,B_10+B_11)
        P_6 = strassen(A_00-A_10,B_00+B_01)
        # Construct and return matrix C
        C = np.zeros((n,n))
        C[:n//2,:n//2] = P_4 + P_3 - P_1 + P_5
        C[:n//2,n//2:] = P_0 + P_1
        C[n//2:,:n//2] = P_2 + P_3
        C[n//2:,n//2:] = P_4 + P_0 - P_2 - P_6
        return C

To check that this works, let's multiply two $8 \times 8$ matrices with random entries both with Strassen's algorithm and with the `@` matrix-multiplication operator:

In [ ]:
# Initialise random number generator
np.random.seed(24157)

n = 8 # Matrix size

# Create two 8x8 matrices with random integer entries in the range [1,...,99]
A = np.random.randint(low=1,high=100,size=(8,8))
B = np.random.randint(low=1,high=100,size=(8,8))

print ("A = ",A)
print ("B = ",B)
print ("A@B = ",A@B)
print ("strassen(A,B) = ",strassen(A,B))


You might want to try this out for different values of $n$. Some unit tests can be found at the very end of this notebook.

### Further improvements and comments
Matrix-matrix multiplication has been optimised further. Coppersmith and Winograd developed an algorithm which is $\mathcal{O}(n^{2.375477})$ in 1990, while Williams (2011) and Le Gall (2014) improved this (slightly) by reducing the complexity to $\mathcal{O}(n^{2.3728639})$. Unfortunately none of these algorithms have any practical relevance.

Finally, it should be kept in mind that many matrices encountered in real-life applications have very special properties. They are often *sparse*, which means that most entries are zero. In this case it is possible to use optimised algorithms which are much faster than the ones we considered here. Efficient algorithms for sparse matrices will be discussed in the final year unit MA40177 (Scientific Computing).

## Check your understanding
This notebook contained some algebra that we skipped over to concentrate on the key ideas. The following pen-and-paper exercises will help you to check whether you understood the material in detail:

* Make sure that you understand what Eq. ($\dagger$) means by writing it down for $n=4$
* Convince yourself that $E_{\text{matrix-matrix}}^{\text{recursive}}(n)$ satisfies the recursion relation in Eq. ($\ddagger$)
* Convince yourself that Eqs. ($\diamond$) and ($\triangleleft$) are equivalent to Eq. ($\dagger$)
* Check that $E_{\text{matrix-matrix}}^{\text{strassen}}(n)$ satisfies the recursion relation in Eq. ($\star$)

## Unit tests for Strassen's algorithm

In [ ]:
def test_1x1():
    n = 1
    A = np.random.randint(low=1,high=100,size=(n,n))
    B = np.random.randint(low=1,high=100,size=(n,n))
    assert np.array_equal(strassen(A,B),A@B)
    
def test_2x2():
    n = 2
    A = np.random.randint(low=1,high=100,size=(n,n))
    B = np.random.randint(low=1,high=100,size=(n,n))
    assert np.array_equal(strassen(A,B),A@B)
    
def test_4x4():
    n = 4
    A = np.random.randint(low=1,high=100,size=(n,n))
    B = np.random.randint(low=1,high=100,size=(n,n))
    assert np.array_equal(strassen(A,B),A@B)
    
def test_64x64():
    n = 64
    A = np.random.randint(low=1,high=100,size=(n,n))
    B = np.random.randint(low=1,high=100,size=(n,n))
    assert np.array_equal(strassen(A,B),A@B)
    
np.random.seed(24157)

run_tests()